In [1]:
import pandas as pd
from pathlib import Path
import holidays
import numpy as np

In [2]:
ROOT = Path.cwd().resolve().parent

DATA = ROOT / "data/toptur"

ANIO = 2026
MES = 5  # Mayo

periodo_evaluado = pd.Period(year=ANIO, month=MES, freq="M")
FECHA_INICIO = periodo_evaluado.start_time.strftime("%Y-%m-%d")
FECHA_FIN = periodo_evaluado.end_time.strftime("%Y-%m-%d")

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

nombre_carpeta_mes = f"{MESES_ES[MES]}{str(ANIO)[-2:]}"  # ej: 'Mayo26'

expediciones = DATA / "Mayo26/expediciones.xls"
frecuencias_fijas = DATA / "POT_V_VALPARAISOUN07_UN07_NORMAL_2026_A1_2.xlsx"

## Calendario

es necesario para evitar no considerar las fechas con 0 expediciones

In [3]:
fechas_rango = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D")
anios = fechas_rango.year.unique().tolist()
feriados_cl = holidays.Chile(years=anios)

def clasificar_tipo_dia(fechas, feriados):
    es_feriado = fechas.to_series(index=range(len(fechas))).apply(lambda x: x in feriados)
    es_domingo = fechas.weekday == 6
    es_sabado = fechas.weekday == 5
    condiciones = [es_feriado.values | es_domingo, es_sabado]
    return np.select(condiciones, ['DF', 'DS'], default='DL')

calendario = pd.DataFrame({"Fecha": fechas_rango})
calendario["tipo_dia"] = clasificar_tipo_dia(fechas_rango, feriados_cl)
calendario.head()

,Fecha,tipo_dia
0,2026-05-01,DF
1,2026-05-02,DS
2,2026-05-03,DF
3,2026-05-04,DL
4,2026-05-05,DL


## Expediciones

In [4]:
df_exp = pd.read_html(expediciones, flavor='lxml')[0]
df_exp["Fecha"] = pd.to_datetime(df_exp["Fecha"])

# tipo_dia de cada expedición observada (usando el mismo criterio que el calendario)
df_exp = df_exp.merge(calendario, on="Fecha", how="left")

df_exp["sentido_str"] = df_exp["Sentido"].astype(str).str.strip().str.upper().str[0]
df_exp["servicio_str"] = df_exp["Servicio"].astype(str).str.strip()
df_exp["periodo_num"] = pd.to_numeric(df_exp["Periodo"], errors="coerce").astype("Int64")

In [5]:
filtro_valida = df_exp["Estado"].astype(str).str.strip() == "Valida"
df_exp_validas = df_exp[filtro_valida].copy()
df_exp_validas.head(2)

,Fecha,Inicio Expedicion,Fin Expedicion,Folio TS,ID Exp,Bus,Chofer,Propietario,Variante,Servicio,...,10,11,12,13,14,15,tipo_dia,sentido_str,servicio_str,periodo_num
2,2026-05-01,2026-05-01 06:59:52,2026-05-01 07:42:29,2605017310420,6886345,116/FRDH12,PRUEBA PRUEBA,GLADYS MONTOLLA .,731,701,...,07:31:51,07:36:12,07:37:48,07:38:46,07:42:29,NaN,DF,I,701,6
3,2026-05-01,2026-05-01 07:00:34,2026-05-01 07:46:32,2605017150420,6886371,153/GWXF15,JUAN LUIS ZUÃIGA BRAVO,NaN,715,705,...,07:32:44,07:35:00,07:39:39,07:41:42,07:45:22,07:46:32,DF,I,705,7


In [6]:
df_conteo = (
    df_exp_validas
    .groupby(['Fecha', 'servicio_str', 'sentido_str', 'tipo_dia', 'periodo_num'])
    .size()
    .reset_index(name="expediciones_observadas")
    .rename(columns={
        'servicio_str': 'servicio',
        'sentido_str': 'sentido',
        'periodo_num': 'periodo'
    })
)
df_conteo.head(2)

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas
0,2026-05-01,701,I,DF,6,1
1,2026-05-01,701,I,DF,7,2


In [7]:
df_conteo[df_conteo["expediciones_observadas"] == 0]

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas


## Frecuencias Establecidas - Anexo 1

In [8]:
excel_completo = pd.ExcelFile(frecuencias_fijas)
hojas_servicios = [hoja for hoja in excel_completo.sheet_names if '-' in hoja]

lista_dataframes = []

for hoja in hojas_servicios:
    # Ej: '701-I' -> servicio='701', sentido='I'
    servicio, sentido = hoja.split('-')

    df_frec = pd.read_excel(excel_completo, sheet_name=hoja, skiprows=10, header=[0, 1])
    df_frec = df_frec.dropna(axis=1, how='all')
    df_frec = df_frec[df_frec.iloc[:, 0].astype(str).str.strip().str.lower() != 'total']

    if df_frec.empty:
        continue

    df_frec.columns = [
        'periodo', 'horario',
        'demanda_DL', 'frecuencia_DL',
        'demanda_DS', 'frecuencia_DS',
        'demanda_DF', 'frecuencia_DF'
    ]

    bloques = []
    for sufijo in ['DL', 'DS', 'DF']:
        bloque = df_frec[['periodo', f'demanda_{sufijo}', f'frecuencia_{sufijo}']].copy()
        bloque['tipo_dia'] = sufijo
        bloque.rename(columns={
            f'demanda_{sufijo}': 'tipo_demanda',
            f'frecuencia_{sufijo}': 'frecuencia'
        }, inplace=True)
        bloques.append(bloque)

    df_hoja_normalizado = pd.concat(bloques, ignore_index=True)
    df_hoja_normalizado['servicio'] = servicio
    df_hoja_normalizado['sentido'] = sentido

    lista_dataframes.append(df_hoja_normalizado)

In [9]:
df_master_frecuencias = pd.concat(lista_dataframes, ignore_index=True)
df_master_frecuencias = df_master_frecuencias[[
    'servicio', 'sentido', 'periodo', 'tipo_dia', 'tipo_demanda', 'frecuencia'
]]
df_master_frecuencias["periodo"] = pd.to_numeric(
    df_master_frecuencias["periodo"], errors='coerce'
    ).astype('Int64')

tipo_demanda viene NaN cuando el Anexo 1 directamente no declara ese periodo/tipo_dia para el servicio (ej. el servicio no opera en ese horario). Esos casos sí se excluyen correctamente, porque no hay exigencia que cumplir. Cualquier periodo con tipo_demanda definido queda incluido incluso si frecuencia fuera 0 para que entre al cálculo.

In [10]:
df_a1 = df_master_frecuencias[df_master_frecuencias["tipo_demanda"].notna()].copy()
df_a1 = df_a1.rename(columns={"frecuencia": "EE"})
df_a1.head(3)

,servicio,sentido,periodo,tipo_dia,tipo_demanda,EE
6,701,I,6,DL,ALTA,8.0
7,701,I,7,DL,ALTA,8.0
8,701,I,8,DL,ALTA,7.0


## Calculo ICF

df_base_exigida cruza todas las fechas del calendario completo (sección 2) con todas las combinaciones servicio/sentido/periodo exigidas por el Anexo 1 para ese tipo_dia. Así, ningún periodo exigido queda fuera del cálculo, sin importar si tuvo o no expediciones registradas.

In [11]:
df_base_exigida = pd.merge(calendario, df_a1, on='tipo_dia', how='left')

df_icf = pd.merge(
    df_base_exigida,
    df_conteo,
    on=['Fecha', 'servicio', 'sentido', 'tipo_dia', 'periodo'],
    how='left'
)

df_icf = df_icf.rename(columns={"expediciones_observadas": "EO"})
df_icf["EO"] = df_icf["EO"].fillna(0)

# ICF normativo: razón EO/EE, capeada en 1.0 (no se premia exceso de oferta)
df_icf['ICF'] = np.floor(np.minimum(df_icf['EE'], df_icf['EO']) / df_icf['EE'] * 100 + 0.5) / 100

### Chequeo de integridad

Expediciones observadas que no calzan con ninguna exigencia del Anexo 1 (útil para detectar errores de codificación de servicio/sentido/periodo, no para el cálculo del ICF).

In [12]:
chequeo = pd.merge(
    df_conteo, df_a1,
    on=['servicio', 'sentido', 'periodo', 'tipo_dia'],
    how='left', indicator=True
)

# Si no hay exigencia asociada (left_only), la frecuencia exigida es 0
chequeo['EE'] = chequeo['EE'].fillna(0)

# Delta de buses de más: expediciones observadas por encima de lo exigido
chequeo['delta_buses_demas'] = chequeo['expediciones_observadas'] - chequeo['EE']

buses_de_mas = chequeo[chequeo['delta_buses_demas'] > 0]

print(f"Periodos con buses de más: {len(buses_de_mas)}")
buses_de_mas[
    ['Fecha', 'periodo', 'servicio', 'sentido', 'tipo_dia',
     'EE', 'expediciones_observadas', 'delta_buses_demas']
].rename(columns={'expediciones_observadas': 'EO'}).sort_values(
    ['servicio', 'sentido', 'Fecha', 'periodo']
).reset_index(drop=True)

Periodos con buses de más: 490


,Fecha,periodo,servicio,sentido,tipo_dia,EE,EO,delta_buses_demas
0,2026-05-01,6,701,I,DF,0.0,1,1.0
1,2026-05-02,6,701,I,DS,5.0,6,1.0
2,2026-05-03,20,701,I,DF,2.0,3,1.0
3,2026-05-04,6,701,I,DL,8.0,9,1.0
4,2026-05-04,8,701,I,DL,7.0,8,1.0
...,...,...,...,...,...,...,...,...
485,2026-05-28,11,706,R,DL,4.0,5,1.0
486,2026-05-29,6,706,R,DL,0.0,2,2.0
487,2026-05-29,12,706,R,DL,4.0,5,1.0
488,2026-05-30,6,706,R,DS,0.0,1,1.0


In [13]:
resumen_delta = (
    buses_de_mas
    .groupby(['servicio', 'sentido'])['delta_buses_demas']
    .sum()
    .reset_index()
    .sort_values('delta_buses_demas', ascending=False)
)
resumen_delta

,servicio,sentido,delta_buses_demas
0,701,I,109.0
1,701,R,109.0
8,705,I,72.0
11,706,R,62.0
7,704,R,53.0
5,703,R,47.0
6,704,I,41.0
2,702,I,28.0
9,705,R,26.0
3,702,R,22.0


In [14]:
periodos_sin_expediciones = df_icf[(df_icf["EE"] > 0) & (df_icf["EO"] == 0)]
periodos_sin_expediciones[['Fecha', 'servicio', 'sentido', 'periodo', 'tipo_dia', 'EE', 'EO', 'ICF']].sort_values(
    ['servicio', 'sentido', 'Fecha', 'periodo']
)

periodos_sin_expediciones

,Fecha,tipo_dia,servicio,sentido,periodo,tipo_demanda,EE,EO,ICF
28,2026-05-01,DF,701,R,21,BAJA,1.0,0.0,0.0
41,2026-05-01,DF,702,I,19,MEDIA,3.0,0.0,0.0
42,2026-05-01,DF,702,I,20,BAJA,2.0,0.0,0.0
48,2026-05-01,DF,702,R,12,MEDIA,3.0,0.0,0.0
53,2026-05-01,DF,702,R,17,MEDIA,3.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
5511,2026-05-31,DF,704,R,21,MEDIA,3.0,0.0,0.0
5525,2026-05-31,DF,705,I,20,BAJA,4.0,0.0,0.0
5540,2026-05-31,DF,705,R,21,BAJA,2.0,0.0,0.0
5554,2026-05-31,DF,706,I,20,MEDIA,3.0,0.0,0.0


## $\psi$

<center>

|meses|$ \psi $|
|---|----|
|<=24|0.90|
|>24|0.95|

In [15]:
def calcular_psi(df, fecha_columna="Fecha", fecha_inicio_operacion=None, mas_de_24_meses=None):
    """
    Calcula psi (ψ) según la antigüedad de la operación (Res. 49/2024, pág. 41):
        - Hasta el mes 24 de operación: psi = 0,90
        - Desde el mes 25 en adelante:  psi = 0,95

    Entregar UNO de los dos parámetros: fecha_inicio_operacion o mas_de_24_meses.
    """
    if fecha_inicio_operacion is None and mas_de_24_meses is None:
        raise ValueError("Debes especificar 'fecha_inicio_operacion' o 'mas_de_24_meses'.")

    if mas_de_24_meses is not None:
        return pd.Series(0.95 if mas_de_24_meses else 0.90, index=df.index)

    fecha_inicio_operacion = pd.Timestamp(fecha_inicio_operacion)

    def mes_operacion(fecha):
        return (fecha.year - fecha_inicio_operacion.year) * 12 + (fecha.month - fecha_inicio_operacion.month) + 1

    meses_operacion = df[fecha_columna].apply(mes_operacion)
    return np.where(meses_operacion <= 24, 0.90, 0.95)

In [16]:
def aplicar_regla_pago(icf, psi=0.95):
    if icf < 0.50:
        return 0.50
    elif icf > psi:
        return 1.00
    else:
        return icf

In [17]:
def tabla_periodo_vs_fecha(df_icf, servicio, sentido):
    """
    Tabla 'periodo x fecha' con la razón cruda EO/EE (sin el cap del ICF normativo) —
    es el reporte de diagnóstico, no el ICF oficial.
    """
    df_filtro = df_icf[
        (df_icf['servicio'] == servicio) & (df_icf['sentido'] == sentido)
    ].copy()

    df_filtro['ratio_crudo'] = df_filtro['EO'] / df_filtro['EE']

    tabla = df_filtro.pivot_table(
        index='periodo', columns='Fecha', values='ratio_crudo', aggfunc='mean'
    )
    tabla.columns = [c.strftime('%Y-%m-%d') for c in tabla.columns]
    tabla['Promedio'] = tabla.mean(axis=1).round(2)
    return tabla

In [18]:
df_icf["psi"] = calcular_psi(df_icf, mas_de_24_meses=True)

In [19]:
df_icf.groupby(["tipo_demanda"])["ICF"].mean().round(2) #este es el resultado

tipo_demanda
ALTA     0.70
BAJA     0.61
MEDIA    0.63
Name: ICF, dtype: float64

In [20]:
df_icf.groupby(["tipo_demanda"])["ICF"].mean().round(2).mean(axis = 0).round(3)

np.float64(0.647)

In [21]:
df_icf.groupby(["tipo_demanda", "servicio"])["ICF"].mean().round(2)

tipo_demanda  servicio
ALTA          701         0.89
              702         0.67
              703         0.60
              704         0.60
              705         0.75
              706         0.69
BAJA          701         0.75
              702         0.39
              703         0.48
              704         0.99
              705         0.71
              706         0.02
MEDIA         701         0.78
              702         0.69
              704         0.51
              705         0.63
              706         0.61
Name: ICF, dtype: float64

In [22]:
df_icf.groupby(["tipo_demanda", "servicio","sentido"])["ICF"].mean().round(2)

tipo_demanda  servicio  sentido
ALTA          701       I          0.89
                        R          0.89
              702       I          0.71
                        R          0.63
              703       I          0.60
                        R          0.59
              704       I          0.61
                        R          0.59
              705       I          0.74
                        R          0.75
              706       I          0.67
                        R          0.70
BAJA          701       I          0.93
                        R          0.73
              702       I          0.46
                        R          0.36
              703       I          0.46
                        R          0.51
              704       I          1.00
                        R          0.98
              705       I          0.66
                        R          0.75
              706       R          0.02
MEDIA         701       I          0.78
        

In [20]:
CARPETA_REPORTES = ROOT / "salidas" / "reporte_servicio_sentido"
CARPETA_REPORTES.mkdir(parents=True, exist_ok=True)

combinaciones = df_icf[['servicio', 'sentido']].drop_duplicates().sort_values(['servicio', 'sentido'])
resumen_tipo_demanda = []

for _, row in combinaciones.iterrows():
    servicio = row['servicio']
    sentido = row['sentido']

    # 1. Tabla detallada periodo x fecha (diagnóstico, ratio crudo)
    tabla = tabla_periodo_vs_fecha(df_icf, servicio=servicio, sentido=sentido)
    tabla.to_excel(CARPETA_REPORTES / f"{servicio}_{sentido}.xlsx")

    # 2. Promedio por tipo_demanda (ICF normativo, capeado con min)
    df_filtro = df_icf[(df_icf['servicio'] == servicio) & (df_icf['sentido'] == sentido)]
    promedios = df_filtro.groupby('tipo_demanda', as_index=False).agg(ICF_promedio=('ICF', 'mean'))
    promedios['ICF_promedio'] = promedios['ICF_promedio'].round(2)
    promedios['servicio'] = servicio
    promedios['sentido'] = sentido

    resumen_tipo_demanda.append(promedios)

df_resumen_tipo_demanda = pd.concat(resumen_tipo_demanda, ignore_index=True)
df_resumen_tipo_demanda = df_resumen_tipo_demanda[['servicio', 'sentido', 'tipo_demanda', 'ICF_promedio']]
df_resumen_tipo_demanda

,servicio,sentido,tipo_demanda,ICF_promedio
0,701,I,ALTA,0.89
1,701,I,BAJA,0.93
2,701,I,MEDIA,0.78
3,701,R,ALTA,0.89
4,701,R,BAJA,0.73
5,701,R,MEDIA,0.80
6,702,I,ALTA,0.71
7,702,I,BAJA,0.46
8,702,I,MEDIA,0.71
9,702,R,ALTA,0.63


In [36]:
df_resumen_por_demanda = (
    df_resumen_tipo_demanda
    .groupby('tipo_demanda', as_index=False)
    .agg(ICF_promedio=('ICF_promedio', 'mean'))
)
df_resumen_por_demanda['ICF_promedio'] = df_resumen_por_demanda['ICF_promedio'].round(2)
df_resumen_por_demanda

,tipo_demanda,ICF_promedio
0,ALTA,0.70
1,BAJA,0.62
2,MEDIA,0.64


In [34]:
df_resumen_tipo_demanda["factor_pago"] = df_resumen_tipo_demanda["ICF_promedio"].apply(
    lambda x: aplicar_regla_pago(x, psi=0.95 if True else 0.90)
)
#el true indica que tiene mas de 24 meses

In [38]:
df_resumen_tipo_demanda.head(2)

,servicio,sentido,tipo_demanda,ICF_promedio,factor_pago
0,701,I,ALTA,0.89,0.89
1,701,I,BAJA,0.93,0.93


In [35]:
df_ranking_final = (
    df_resumen_tipo_demanda
    .groupby("servicio")["factor_pago"]
    .mean()
    .reset_index()
    .rename(columns={"factor_pago": "ICF"})
)
df_ranking_final["ICF"] = df_ranking_final["ICF"].round(2)
df_ranking_final

,servicio,ICF
0,701,0.84
1,702,0.62
2,703,0.55
3,704,0.70
4,705,0.69
5,706,0.62
